# WikiArt Style Classification with PyTorch

In this beginner-friendly tutorial, we will build an AI model that predicts the **artistic style** of a painting (for example Impressionism, Cubism, Realism) using the WikiArt dataset.

We will use a **pretrained CNN** (ResNet-18) from `torchvision` and fine-tune it for our style classification task.

## 1. Project Overview

Our goal is simple:
- Read the WikiArt training and validation CSV files
- Load images and labels with a custom PyTorch `Dataset`
- Train a pretrained CNN to predict style labels
- Evaluate accuracy on validation data
- Make a prediction for one image

The dataset has **train and validation splits only**. We will use validation for monitoring. Optionally, we can split a small part of validation to simulate a test set.

## 2. Import Libraries

First, we import the tools we need for data loading, image preprocessing, model training, and evaluation.

In [1]:
from pathlib import Path
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

# Avoid noisy warnings for very large source images; we resize them immediately
warnings.simplefilter("ignore", Image.DecompressionBombWarning)

# Make results more reproducible for this tutorial
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 3. Locate the WikiArt Dataset Automatically

To keep this notebook flexible, we will search for the dataset folder and CSV files inside the project directory instead of hardcoding exact paths.

In [2]:
def find_project_root(start: Path) -> Path:
    """Walk upward until we find a folder that looks like the project root."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datasets").exists() and (candidate / "README.md").exists():
            return candidate
    return start.resolve()

project_root = find_project_root(Path.cwd())
wikiart_dir = project_root / "datasets" / "Wikiart"

if not wikiart_dir.exists():
    raise FileNotFoundError(f"Could not find WikiArt directory at: {wikiart_dir}")

print(f"Project root: {project_root}")
print(f"WikiArt dir:  {wikiart_dir}")

Project root: C:\Users\Thijs\Desktop\Ai Art Critic
WikiArt dir:  C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart


### Understand the CSV format

The WikiArt style CSV files use rows like:
`style_folder/image_name.jpg,label_id`

There is no header row, so we will load with custom column names.

In [3]:
def find_split_csv(data_dir: Path, split: str) -> Path:
    """Find a CSV for a split (train/val) with preference for style-related filenames."""
    split = split.lower()
    csvs = sorted(data_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    preferred = [p for p in csvs if split in p.stem.lower() and "style" in p.stem.lower()]
    if preferred:
        return preferred[0]

    fallback = [p for p in csvs if split in p.stem.lower()]
    if fallback:
        return fallback[0]

    raise FileNotFoundError(f"Could not find a '{split}' CSV in {data_dir}")

train_csv = find_split_csv(wikiart_dir, "train")
val_csv = find_split_csv(wikiart_dir, "val")

print(f"Train CSV: {train_csv.name}")
print(f"Val CSV:   {val_csv.name}")

train_df = pd.read_csv(train_csv, header=None, names=["relative_path", "label"])
val_df = pd.read_csv(val_csv, header=None, names=["relative_path", "label"])

print("\nTrain sample:")
display(train_df.head())
print(f"Train rows: {len(train_df):,}")
print(f"Val rows:   {len(val_df):,}")

Train CSV: style_train.csv
Val CSV:   style_val.csv

Train sample:


,relative_path,label
0,Impressionism/edgar-degas_landscape-on-the-orn...,12
1,Realism/camille-corot_mantes-cathedral.jpg,21
2,Abstract_Expressionism/gene-davis_untitled-197...,0
3,Symbolism/kuzma-petrov-vodkin_in-the-1920.jpg,24
4,Impressionism/maurice-prendergast_paris-boulev...,12


Train rows: 57,025
Val rows:   24,421


## 4. Prepare Label Mapping

The label is numeric, but each image path includes the style folder name.

We will build a mapping from label ID to style name so our predictions are easier to understand.

In [4]:
def extract_style_name(relative_path: str) -> str:
    return Path(relative_path).parts[0]

train_df["style_name"] = train_df["relative_path"].map(extract_style_name)
val_df["style_name"] = val_df["relative_path"].map(extract_style_name)

label_to_style = (
    train_df[["label", "style_name"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["style_name"]
    .to_dict()
)

num_classes = len(label_to_style)
print(f"Number of classes: {num_classes}")
print("First 10 label -> style mappings:")
for label, style in list(label_to_style.items())[:10]:
    print(f"{label:2d} -> {style}")

Number of classes: 27
First 10 label -> style mappings:
 0 -> Abstract_Expressionism
 1 -> Action_painting
 2 -> Analytical_Cubism
 3 -> Art_Nouveau_Modern
 4 -> Baroque
 5 -> Color_Field_Painting
 6 -> Contemporary_Realism
 7 -> Cubism
 8 -> Early_Renaissance
 9 -> Expressionism


## 5. Prepare Image Transformations

Neural networks need consistent input size and scale. We will:
- resize each image
- convert image to tensor
- normalize with ImageNet mean/std (good practice for pretrained models)

In [5]:
image_size = 224

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 6. Create a Custom Dataset Loader

Now we define a custom PyTorch `Dataset`.

Each item will return:
- transformed image tensor
- numeric label

In [6]:
class WikiArtStyleDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_root: Path, transform=None, max_retries: int = 10):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        last_error = None

        # Retry with a different sample if a file is missing/corrupt.
        for _ in range(self.max_retries):
            row = self.df.iloc[idx]
            img_path = self.image_root / row["relative_path"]
            label = int(row["label"])
            try:
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                return image, label
            except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as exc:
                last_error = exc
                idx = random.randint(0, len(self.df) - 1)

        raise RuntimeError(f"Failed to load image after {self.max_retries} retries. Last error: {last_error}")

## 7. Create DataLoaders

We will build DataLoaders for training and validation.

Because this dataset has no test split, we can optionally split a small part of validation into a simulated test set.

In [7]:
batch_size = 32
num_workers = 0  # Keep 0 for beginner-friendly compatibility on Windows

def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    full_paths = df["relative_path"].map(lambda p: image_root / p)
    exists_mask = full_paths.map(lambda p: p.exists())

    removed = int((~exists_mask).sum())
    kept = int(exists_mask.sum())
    print(f"{split_name}: kept {kept:,} rows, removed {removed:,} missing files")

    cleaned = df.loc[exists_mask].reset_index(drop=True)
    if cleaned.empty:
        raise RuntimeError(f"No valid images left in {split_name} after filtering missing files.")
    return cleaned

train_df_clean = filter_existing_rows(train_df, wikiart_dir, "train")
val_df_clean = filter_existing_rows(val_df, wikiart_dir, "validation")

train_dataset = WikiArtStyleDataset(train_df_clean, wikiart_dir, transform=train_transform)
val_dataset_full = WikiArtStyleDataset(val_df_clean, wikiart_dir, transform=val_transform)

CREATE_TEST_SPLIT = True
TEST_FRACTION_FROM_VAL = 0.2

if CREATE_TEST_SPLIT and len(val_dataset_full) > 10:
    test_size = int(len(val_dataset_full) * TEST_FRACTION_FROM_VAL)
    val_size = len(val_dataset_full) - test_size

    val_dataset, test_dataset = random_split(
        val_dataset_full,
        [val_size, test_size],
        generator=torch.Generator().manual_seed(SEED),
    )
else:
    val_dataset = val_dataset_full
    test_dataset = None

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = (
    DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    if test_dataset is not None
    else None
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
if test_loader is not None:
    print(f"Test batches:  {len(test_loader)} (simulated from validation)")
else:
    print("Test loader:   not created")

train: kept 57,023 rows, removed 2 missing files
validation: kept 24,421 rows, removed 0 missing files
Train batches: 1782
Val batches:   611
Test batches:  153 (simulated from validation)


## 8. Build the Model

We load a pretrained **ResNet-18** model and replace its final classification layer so it predicts the number of WikiArt styles.

In [8]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(model.fc)

Linear(in_features=512, out_features=27, bias=True)


## 9. Train the Model

This training loop runs for a few epochs, updates model weights on training data, and checks validation performance after each epoch.

In [9]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

epochs = 3
history = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer=None)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

history_df = pd.DataFrame(history)
display(history_df)

Epoch 01/3 | train_loss=1.5919, train_acc=0.480 | val_loss=1.3446, val_acc=0.547
Epoch 02/3 | train_loss=1.2057, train_acc=0.590 | val_loss=1.2716, val_acc=0.574
Epoch 03/3 | train_loss=0.9913, train_acc=0.663 | val_loss=1.2548, val_acc=0.578


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,1.591934,0.479736,1.344574,0.547116
1,2,1.205745,0.590095,1.271594,0.573681
2,3,0.991275,0.662803,1.254777,0.578185


## 10. Evaluate the Model

Now we calculate final accuracy on validation data.

If we created a simulated test split from validation, we also report that.

In [10]:
def evaluate_accuracy(model, loader):
    model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    return total_correct / total_samples

val_accuracy = evaluate_accuracy(model, val_loader)
print(f"Validation accuracy: {val_accuracy:.3f}")

if test_loader is not None:
    test_accuracy = evaluate_accuracy(model, test_loader)
    print(f"Simulated test accuracy: {test_accuracy:.3f}")

Validation accuracy: 0.578
Simulated test accuracy: 0.577


## 11. Make a Prediction for One Image

Finally, we load a single image, run inference, and convert the predicted numeric label back to a style name.

In [11]:
def predict_image_style(model, image_path: Path, transform, label_to_style_map):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        pred_label = int(outputs.argmax(dim=1).item())

    pred_style = label_to_style_map.get(pred_label, f"Unknown label {pred_label}")
    return pred_label, pred_style

sample_relative_path = val_df.iloc[0]["relative_path"]
sample_image_path = wikiart_dir / sample_relative_path

pred_label, pred_style = predict_image_style(model, sample_image_path, val_transform, label_to_style)
print(f"Image: {sample_relative_path}")
print(f"Predicted label: {pred_label}")
print(f"Predicted style: {pred_style}")

Image: Impressionism/edgar-degas_dancers-on-set-1880.jpg
Predicted label: 12
Predicted style: Impressionism


## 12. Next Steps

Great work. You now have a complete beginner pipeline for art style classification.

Possible improvements:
- Train for more epochs and tune learning rate
- Add stronger data augmentation
- Save and load model checkpoints
- Track metrics like precision/recall per class

To integrate this into a web app later, you can expose the trained model through a backend API. Users upload an artwork image, your API preprocesses it, runs inference, and returns the predicted style name to the frontend.